# A. 샤프트 비교 데이터셋 1회 준비

다겸님이 **한 번만** 실행하는 노트북입니다. 원본 161장 → 가로 2992px → 기존 `best.pt` pre-annotation → 사람 검수 → 고정 데이터셋 게시 순서입니다.

> `dataset/`이 이미 있으면 이 노트북은 덮어쓰지 않습니다. 운영 가중치도 읽기만 합니다. `._` 파일은 전 단계에서 제외합니다.

검수 기준 문장: **좌우 = 목 구간이 시작·끝나는 단차 안쪽 끝, 상하 = 샤프트 외곽선에 맞춤.**


In [ ]:
# 0) 환경과 설정 — 이 노트북을 담당한 사람만 경로를 맞춥니다.
!pip install -q -U ultralytics google-api-python-client
from pathlib import Path
import csv, json, shutil, sys
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = Path('/content/drive/MyDrive/shaft_sweep')
# 권장: 공유 폴더 바로가기의 실제 경로. 없으면 로그인된 계정으로 Drive API를 사용합니다.
SOURCE_DIR = Path('/content/drive/MyDrive/샤프트_원본_161장')
GDRIVE_FOLDER_ID = '1BbHHsoB8PDg0Sm1GQMl9tDtAF00VtHvK'
# best.pt는 아래 후보에서 자동 탐색합니다. 없으면 Colab 업로드 창이 열립니다.
BEST_PT_CANDIDATES = [Path('/content/best.pt'), DRIVE/'source_weights'/'best.pt', Path('/content/drive/MyDrive/best.pt')]
CALIB_SOURCE_DIR = DRIVE / 'calib_source'  # 기준부품 원본 이미지를 넣는 폴더
STAGE = DRIVE / 'dataset_staging'
FINAL = DRIVE / 'dataset'
TARGET_WIDTH, CONF, IMGSZ = 2992, 0.25, 640
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def images_in(folder):
    return sorted(p for p in folder.rglob('*') if p.is_file() and not p.name.startswith('._') and p.suffix.lower() in IMAGE_SUFFIXES)

if FINAL.exists():
    raise RuntimeError('고정 dataset/이 이미 있습니다. 절대 덮어쓰지 마세요. 기존 폴더를 팀과 확인하세요.')
BEST_PT = next((p for p in BEST_PT_CANDIDATES if p.exists()), None)
if BEST_PT is None:
    try:
        from google.colab import files
        print('기존 운영 가중치 best.pt를 선택해 업로드하세요. 업로드본은 현재 Colab 세션에서 읽기만 합니다.')
        uploaded = files.upload()
        pt_files = [Path(name).resolve() for name in uploaded if Path(name).suffix.lower()=='.pt']
        if len(pt_files) != 1:
            raise FileNotFoundError('한 번에 best.pt 파일 하나만 선택해야 합니다.')
        BEST_PT = pt_files[0]
    except ImportError:
        raise FileNotFoundError('best.pt가 없습니다. /content/best.pt 또는 DRIVE/source_weights/best.pt에 두세요.')
print('사용할 가중치:', BEST_PT)
for name in ['images', 'labels', 'calib', 'previews']:
    (STAGE / name).mkdir(parents=True, exist_ok=True)
print('준비 완료:', STAGE)


In [ ]:
# 1) 원본 확보 + 2992px 리사이즈
def download_private_drive_folder(folder_id, output_dir):
    """gdown이 못 받는 비공개 공유 폴더를 현재 로그인 계정 권한으로 받습니다."""
    import io, logging
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)  # timeout 관련 무해한 경고 숨김
    auth.authenticate_user()
    service = build('drive', 'v3', cache_discovery=False)
    output_dir.mkdir(parents=True, exist_ok=True)
    downloaded = 0

    def children(parent_id):
        token = None
        while True:
            response = service.files().list(
                q=f"'{parent_id}' in parents and trashed=false",
                fields='nextPageToken, files(id,name,mimeType,size)',
                pageSize=1000, pageToken=token,
                supportsAllDrives=True, includeItemsFromAllDrives=True,
            ).execute(num_retries=3)
            yield from response.get('files', [])
            token = response.get('nextPageToken')
            if not token: break

    def visit(parent_id, local_dir):
        nonlocal downloaded
        local_dir.mkdir(parents=True, exist_ok=True)
        for item in children(parent_id):
            target = local_dir / item['name']
            if item['mimeType'] == 'application/vnd.google-apps.folder':
                visit(item['id'], target)
                continue
            if item['mimeType'].startswith('application/vnd.google-apps.'):
                print('Google 문서 형식은 건너뜀:', item['name'])
                continue
            expected_size = int(item.get('size', -1))
            if target.exists() and expected_size >= 0 and target.stat().st_size == expected_size:
                downloaded += 1
                continue
            request = service.files().get_media(fileId=item['id'], supportsAllDrives=True)
            with target.open('wb') as fh:
                loader = MediaIoBaseDownload(fh, request, chunksize=16*1024*1024)
                done = False
                while not done:
                    _, done = loader.next_chunk(num_retries=3)
            downloaded += 1
            if downloaded % 20 == 0: print(f'  다운로드 {downloaded}개 완료')
    visit(folder_id, output_dir)
    return downloaded

mounted_images = images_in(SOURCE_DIR) if SOURCE_DIR.exists() else []
if len(mounted_images) >= 161:
    raw_dir = SOURCE_DIR
    print('MyDrive 바로가기에서 원본을 읽습니다:', raw_dir)
else:
    raw_dir = Path('/content/shaft_raw_download')
    try:
        count = download_private_drive_folder(GDRIVE_FOLDER_ID, raw_dir)
        print('Drive API 다운로드 완료:', count, '개 파일')
    except Exception as exc:
        raise RuntimeError(
            '공유 폴더 다운로드에 실패했습니다. Colab 인증 계정이 원본 폴더에 접근 가능한지 확인하거나, '
            'Drive 웹에서 해당 폴더를 “내 드라이브에 바로가기 추가”한 뒤 SOURCE_DIR을 그 경로로 바꾸세요. '
            f'원인: {type(exc).__name__}: {exc}'
        ) from exc

all_raw_images = images_in(raw_dir)
jpeg_images = [p for p in all_raw_images if p.suffix.lower() in {'.jpg', '.jpeg'}]
# 이 폴더의 실측표 PNG는 학습 사진이 아니므로, JPEG 161장이 맞으면 PNG를 제외합니다.
raw_images = jpeg_images if len(jpeg_images) == 161 else all_raw_images
assert len(raw_images) == 161, (
    f'원본 사진은 정확히 161장이어야 합니다. JPEG={len(jpeg_images)}, 전체 이미지={len(all_raw_images)}. '
    '누락 여부와 SOURCE_DIR을 확인하세요.'
)
assert len({p.name for p in raw_images}) == 161, '서로 다른 폴더에 같은 파일명이 있습니다. 게시 전 고유 이름으로 정리하세요.'

def resize_2992(src, dst):
    image = cv2.imread(str(src))
    if image is None: raise ValueError(f'읽을 수 없는 이미지: {src}')
    h, w = image.shape[:2]
    new_h = round(h * TARGET_WIDTH / w)
    interpolation = cv2.INTER_AREA if w > TARGET_WIDTH else cv2.INTER_CUBIC
    resized = cv2.resize(image, (TARGET_WIDTH, new_h), interpolation=interpolation)
    if not cv2.imwrite(str(dst), resized): raise IOError(f'저장 실패: {dst}')

for i, src in enumerate(raw_images, 1):
    dst = STAGE / 'images' / src.name
    if not dst.exists(): resize_2992(src, dst)
print('리사이즈 완료:', len(images_in(STAGE / 'images')), '장')

# 캘리브레이션은 실측 20.021mm 기준부품 사진입니다. 원본 161장과 별도로 보관합니다.
existing_calib = images_in(STAGE / 'calib')
if existing_calib:
    print('기존 staging 캘리브레이션 이미지를 재사용합니다:', len(existing_calib), '장')
else:
    calib_sources = images_in(CALIB_SOURCE_DIR) if CALIB_SOURCE_DIR.exists() else []
    if not calib_sources:
        try:
            from google.colab import files
            print('캘리브레이션 기준부품(실측 20.021mm) 이미지를 선택하세요. sample_07.jpg 1장 이상이 필요합니다.')
            uploaded = files.upload()
            calib_sources = [
                Path(name).resolve() for name in uploaded
                if not Path(name).name.startswith('._') and Path(name).suffix.lower() in IMAGE_SUFFIXES
            ]
        except ImportError:
            pass
    if not calib_sources:
        raise FileNotFoundError(
            '캘리브레이션 이미지가 없습니다. 실측 20.021mm 기준부품 사진을 '
            'MyDrive/shaft_sweep/calib_source/에 넣거나 업로드 창에서 선택하세요.'
        )
    if len({p.name for p in calib_sources}) != len(calib_sources):
        raise ValueError('캘리브레이션 이미지 파일명이 중복됩니다.')
    for src in calib_sources:
        dst = STAGE / 'calib' / src.name
        if not dst.exists(): resize_2992(src, dst)
    existing_calib = images_in(STAGE / 'calib')
print('캘리브레이션 준비 완료:', len(existing_calib), '장')


In [ ]:
# 2) 기존 best.pt pre-annotation — 모든 박스를 CSV에 남기고 top-1만 YOLO 라벨로 저장
model = YOLO(str(BEST_PT))
detections, tops = [], []
for image_path in images_in(STAGE / 'images'):
    image = cv2.imread(str(image_path))
    h, w = image.shape[:2]
    result = model.predict(str(image_path), conf=CONF, imgsz=IMGSZ, verbose=False)[0]
    boxes = []
    if result.boxes is not None:
        for box in result.boxes:
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(float, box.xyxy[0].cpu().numpy())
            boxes.append(dict(conf=conf, x1=x1, y1=y1, x2=x2, y2=y2, box_w=x2-x1, box_h=y2-y1, center_x_pct=100*(x1+x2)/(2*w)))
    boxes.sort(key=lambda b: b['conf'], reverse=True)
    for rank, box in enumerate(boxes, 1): detections.append({'image': image_path.name, 'detection_rank': rank, **box})
    if boxes:
        b = boxes[0]
        xc, yc = (b['x1']+b['x2'])/(2*w), (b['y1']+b['y2'])/(2*h)
        bw, bh = b['box_w']/w, b['box_h']/h
        label_path = STAGE/'labels'/f'{image_path.stem}.txt'
        if not label_path.exists(): label_path.write_text(f'0 {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}\n')  # 재실행 시 사람 수정 라벨 보존
        tops.append({'image': image_path.name, 'n_detections': len(boxes), 'top1_conf': b['conf'], 'top2_conf': boxes[1]['conf'] if len(boxes)>1 else np.nan, **b})
    else:
        label_path = STAGE/'labels'/f'{image_path.stem}.txt'
        if not label_path.exists(): label_path.write_text('')
        tops.append({'image': image_path.name, 'n_detections': 0, 'top1_conf': np.nan, 'top2_conf': np.nan, 'box_w': np.nan, 'box_h': np.nan, 'center_x_pct': np.nan})

top_df = pd.DataFrame(tops)
med_w, med_h, med_cx = top_df['box_w'].median(), top_df['box_h'].median(), top_df['center_x_pct'].median()
def reasons(row):
    why = []
    if row.n_detections == 0: why.append('no_detection')
    if pd.notna(row.top1_conf) and row.top1_conf < .80: why.append('top1_conf<0.80')
    if pd.notna(row.top2_conf) and row.top1_conf-row.top2_conf < .15: why.append('top1_top2_gap<0.15')
    if pd.notna(row.box_w) and not (.9*med_w <= row.box_w <= 1.1*med_w): why.append('box_w_outside_10pct')
    if pd.notna(row.box_h) and not (.9*med_h <= row.box_h <= 1.1*med_h): why.append('box_h_outside_10pct')
    if pd.notna(row.center_x_pct) and abs(row.center_x_pct-med_cx) > 3: why.append('center_x_outside_3pp')
    return ';'.join(why)
top_df['review_reason'] = top_df.apply(reasons, axis=1)
top_df['review_flag'] = top_df['review_reason'].ne('')
det_df = pd.DataFrame(detections)
if len(det_df): det_df = det_df.merge(top_df[['image','n_detections','review_flag','review_reason']], on='image', how='left')
# 검출 0장도 보고서에서 사라지지 않도록 별도 한 행 추가
missing = top_df.loc[top_df.n_detections.eq(0), ['image','n_detections','review_flag','review_reason']]
report = pd.concat([det_df, missing], ignore_index=True, sort=False)
report.to_csv(STAGE/'prelabel_report.csv', index=False, encoding='utf-8-sig')
top_df.to_csv(STAGE/'prelabel_summary.csv', index=False, encoding='utf-8-sig')
print('검출 0장:', int((top_df.n_detections==0).sum()), '/ 검수 우선:', int(top_df.review_flag.sum()), '/ 2개 이상 검출:', int((top_df.n_detections>=2).sum()))


In [ ]:
# 3) 확대 프리뷰: top-1 박스와 박스 폭의 18~82% 측정 경계를 표시
for row in top_df.itertuples(index=False):
    image = cv2.imread(str(STAGE/'images'/row.image))
    if row.n_detections == 0:
        preview = image
        cv2.putText(preview, 'NO DETECTION - REVIEW', (40,80), cv2.FONT_HERSHEY_SIMPLEX, 2, (0,0,255), 5)
    else:
        x1,y1,x2,y2 = map(int, [row.x1,row.y1,row.x2,row.y2])
        bw, bh = x2-x1, y2-y1
        mx, my = int(.15*bw), int(.5*bh)
        cx1,cx2 = max(0,x1-mx), min(image.shape[1],x2+mx)
        cy1,cy2 = max(0,y1-my), min(image.shape[0],y2+my)
        preview = image[cy1:cy2, cx1:cx2].copy()
        a,b,c,d = x1-cx1,y1-cy1,x2-cx1,y2-cy1
        cv2.rectangle(preview,(a,b),(c,d),(0,255,0),4)
        for frac in (.18,.82):
            xx = a + int((c-a)*frac); cv2.line(preview,(xx,b),(xx,d),(255,0,255),3)
        color = (0,0,255) if row.review_flag else (0,255,0)
        cv2.putText(preview, f'conf={row.top1_conf:.3f} n={row.n_detections} review={row.review_flag}', (20,45), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)
    cv2.imwrite(str(STAGE/'previews'/f'{Path(row.image).stem}.jpg'), preview)
print('프리뷰 저장:', STAGE/'previews')
print('먼저 review_flag=True를 보고, 이어서 전 프리뷰를 훑으세요. 수정 시 같은 시편 그룹 전체를 같은 기준으로 보정하세요.')


## 사람 검수 구간 (10~20분)

1. `prelabel_summary.csv`에서 `review_flag=True`를 먼저 확인합니다.
2. `previews/` 전체를 훑고 라벨이 기준 문장을 벗어난 경우 `labels/*.txt`를 수정합니다.
3. 아래 셀이 만든 `specimen_map.csv`에 실제 `specimen_id`, `is_ng`, `true_mm`를 입력합니다. 파일명으로 추측하지 않습니다.
4. top-2가 다른 구간을 잡았는지 확인해 `REVIEW_NOTES`에 기록합니다.
5. 검수 완료 후 마지막 셀의 `REVIEW_COMPLETE=True`로 바꾸어 게시합니다.


In [ ]:
# 4) specimen_map 템플릿 — 빈 칸은 사람이 실측 기록을 보고 채웁니다.
map_path = STAGE/'specimen_map.csv'
if not map_path.exists():
    pd.DataFrame({'image':[p.name for p in images_in(STAGE/'images')], 'specimen_id':'', 'is_ng':'', 'true_mm':''}).to_csv(map_path, index=False, encoding='utf-8-sig')
print('편집할 파일:', map_path)
print('주의: specimen_id는 파일명 추측이 아니라 실제 촬영/시편 기록을 기준으로 입력하세요.')


In [ ]:
# 5) 최종 검증 후 불변 dataset/ 게시 — 검수 전에 True로 바꾸지 마세요.
REVIEW_COMPLETE = False
REVIEW_NOTES = 'top-2 좌표 육안 확인 결과를 여기에 기록하세요.'
if not REVIEW_COMPLETE:
    raise RuntimeError('사람 검수가 아직 완료되지 않았습니다. 검수 후 REVIEW_COMPLETE=True로 바꾸세요.')

mapping = pd.read_csv(STAGE/'specimen_map.csv')
required = ['image','specimen_id','is_ng','true_mm']
assert list(mapping.columns) == required, f'컬럼은 정확히 {required}여야 합니다.'
assert len(mapping)==161 and not mapping[required].isna().any().any(), '161행의 모든 값을 채우세요.'
assert mapping.image.nunique()==161, 'image가 중복됐습니다.'
assert set(mapping.image)=={p.name for p in images_in(STAGE/'images')}, '이미지 목록과 CSV가 다릅니다.'
assert mapping.specimen_id.astype(str).nunique()==13, '시편은 정확히 13개여야 합니다.'
ng = mapping.is_ng.astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
assert not ng.isna().any(), 'is_ng는 True/False 또는 1/0으로 쓰세요.'
spec_ng = pd.DataFrame({'specimen_id':mapping.specimen_id.astype(str),'is_ng':ng}).groupby('specimen_id').is_ng.agg(['first','nunique'])
assert (spec_ng['nunique']==1).all(), '같은 시편 안의 is_ng가 서로 다릅니다.'
assert int(spec_ng['first'].sum())==3 and int((~spec_ng['first']).sum())==10, '정상 10개/불량 3개여야 합니다.'
counts = mapping.groupby(mapping.specimen_id.astype(str)).size()
assert sorted(counts.tolist()) == [7,7,7]+[14]*10, f'촬영 장수는 정상 10×14, 불량 3×7이어야 합니다: {counts.to_dict()}'
mapping['true_mm'] = pd.to_numeric(mapping.true_mm, errors='raise')
assert (mapping.groupby(mapping.specimen_id.astype(str)).true_mm.nunique()==1).all(), '같은 시편 안의 true_mm가 서로 다릅니다.'
labels = sorted((STAGE/'labels').glob('*.txt'))
assert len(labels)==161, f'라벨이 {len(labels)}개입니다.'
for label in labels:
    lines = [x for x in label.read_text().splitlines() if x.strip()]
    assert len(lines)==1 and len(lines[0].split())==5 and lines[0].split()[0]=='0', f'라벨은 class 0 박스 1개여야 합니다: {label.name}'
assert images_in(STAGE/'calib'), 'calib 이미지가 없습니다.'

readme = f'''# shaft_sweep 고정 데이터셋

- 이미지: 161장, 가로 2992px
- 시편: 정상 10개×14장 + 불량 3개×7장 = 13개
- 라벨 출처: 기존 운영 best.pt(conf=0.25, imgsz=640) pre-annotation 후 사람 검수
- 박스 기준: 좌우 = 목 구간이 시작·끝나는 단차 안쪽 끝, 상하 = 샤프트 외곽선
- 주의: 이 라벨은 기존 박스 규칙의 옳음을 독립적으로 검증하는 ground truth가 아니다.
- top-2 확인 기록: {REVIEW_NOTES}
- 2개 이상 검출된 이미지: {int((top_df.n_detections>=2).sum())}/161
'''
(STAGE/'README.md').write_text(readme, encoding='utf-8')
if FINAL.exists(): raise RuntimeError('dataset/이 실행 중 새로 생겼습니다. 덮어쓰지 않고 중단합니다.')
# previews와 summary도 감사 추적용으로 함께 보존하되 B는 images/labels/calib/map만 읽습니다.
shutil.copytree(STAGE, FINAL)
print('게시 완료:', FINAL)
print('이제 팀에 “MyDrive/shaft_sweep/dataset을 읽으세요. 이 폴더는 수정 금지입니다.”라고 공지하세요.')
